# Eat Safe, Love

## Notebook Set Up

In [1]:
from pymongo import MongoClient
import pandas as pd
from pprint import pprint

In [2]:
# Create an instance of MongoClient
mongo = MongoClient(port=27017)

In [3]:
# assign the uk_food database to a variable name
db = mongo['uk_food']

In [4]:
# review the collections in our database
print(db.list_collection_names())

['establishments']


In [5]:
# assign the collection to a variable
establishments = db['establishments']

## Part 3: Exploratory Analysis
Unless otherwise stated, for each question: 
* Use `count_documents` to display the number of documents contained in the result.
* Display the first document in the results using `pprint`.
* Convert the result to a Pandas DataFrame, print the number of rows in the DataFrame, and display the first 10 rows.

### 1. Which establishments have a hygiene score equal to 20?

In [10]:
# Find the establishments with a hygiene score of 20
query = {'scores.Hygiene': 20}
results = establishments.find(query)
# Use count_documents to display the number of documents in the result
count = establishments.count_documents(query)
# Display the first document in the results using pprint
first_document = results.limit(1)
for x in first_document:
    pprint(x)

{'AddressLine1': '5-6 Southfields Road',
 'AddressLine2': 'Eastbourne',
 'AddressLine3': 'East Sussex',
 'AddressLine4': '',
 'BusinessName': 'The Chase Rest Home',
 'BusinessType': 'Caring Premises',
 'BusinessTypeID': 5,
 'ChangesByServerID': 0,
 'Distance': 4613.888288172291,
 'FHRSID': 110681,
 'LocalAuthorityBusinessID': '4029',
 'LocalAuthorityCode': '102',
 'LocalAuthorityEmailAddress': 'Customerfirst@eastbourne.gov.uk',
 'LocalAuthorityName': 'Eastbourne',
 'LocalAuthorityWebSite': 'http://www.eastbourne.gov.uk/foodratings',
 'NewRatingPending': False,
 'Phone': '',
 'PostCode': 'BN21 1BU',
 'RatingDate': '2021-09-23T00:00:00',
 'RatingKey': 'fhrs_0_en-gb',
 'RatingValue': 0,
 'RightToReply': '',
 'SchemeType': 'FHRS',
 '_id': ObjectId('67a1544eb8342c0e3307ede1'),
 'geocode': {'latitude': Decimal128('50.769705'),
             'longitude': Decimal128('0.27694')},
 'links': [{'href': 'https://api.ratings.food.gov.uk/establishments/110681',
            'rel': 'self'}],
 'meta': {'

In [22]:
# Convert the result to a Pandas DataFrame
hygiene_df = pd.DataFrame(results)
# Display the number of rows in the DataFrame
print('Number of Rows:', len(df))
# Display the first 10 rows of the DataFrame
hygiene_df.head(10)

Number of Rows: 0


""


### 2. Which establishments in London have a `RatingValue` greater than or equal to 4?

In [34]:
# Find the establishments with London as the Local Authority and has a RatingValue greater than or equal to 4.
london_query = {'LocalAuthorityName': 'London', 'RatingValue': {'$gte': 4}}
london_results = establishments.find(london_query)

# Use count_documents to display the number of documents in the result
print('Number of Documents:', establishments.count_documents(london_query))
# Display the first document in the results using pprint
for x in london_results:
    print(x)

Number of Documents: 0


In [36]:
# Convert the result to a Pandas DataFrame
ratingvalue_df = pd.DataFrame(results)
# Display the number of rows in the DataFrame
print('Number of Rows:', len(ratingvalue_df))
# Display the first 10 rows of the DataFrame
ratingvalue_df.head(10)

Number of Rows: 0


""


### 3. What are the top 5 establishments with a `RatingValue` rating value of 5, sorted by lowest hygiene score, nearest to the new restaurant added, "Penang Flavours"?

In [23]:
# Search within 0.01 degree on either side of the latitude and longitude.
# Rating value must equal 5
# Sort by hygiene score

degree_search = 0.01
latitude = 51.49014200
longitude = 0.08384000

query = {'RatingValue': 5,
    'geocode.latitude': {'$gte': latitude - degree_search, '$lte': latitude + degree_search},
    'geocode.longitude': {'$gte': longitude - degree_search, '$lte': longitude + degree_search}
    }
sort = [('scores.Hygiene', -1)]
limit = 5
projection = None

hygiene_results = establishments.find(query, projection).sort(sort).limit(limit)
# Print the results
for x in hygiene_results:
    pprint(x)

{'AddressLine1': '101 Plumstead High Street',
 'AddressLine2': '',
 'AddressLine3': 'Plumstead',
 'AddressLine4': 'Greenwich',
 'BusinessName': 'Lucky Food & Wine',
 'BusinessType': 'Retailers - other',
 'BusinessTypeID': 4613,
 'ChangesByServerID': 0,
 'Distance': 4647.024793263386,
 'FHRSID': 695287,
 'LocalAuthorityBusinessID': 'PI/000182135',
 'LocalAuthorityCode': '511',
 'LocalAuthorityEmailAddress': 'health@royalgreenwich.gov.uk',
 'LocalAuthorityName': 'Greenwich',
 'LocalAuthorityWebSite': 'http://www.royalgreenwich.gov.uk',
 'NewRatingPending': False,
 'Phone': '',
 'PostCode': 'SE18 1SB',
 'RatingDate': '2022-06-25T00:00:00',
 'RatingKey': 'fhrs_5_en-gb',
 'RatingValue': 5,
 'RightToReply': '',
 'SchemeType': 'FHRS',
 '_id': ObjectId('67a1544fb8342c0e330846e0'),
 'geocode': {'latitude': Decimal128('51.4878934'),
             'longitude': Decimal128('0.0910104')},
 'links': [{'href': 'http://api.ratings.food.gov.uk/establishments/695287',
            'rel': 'self'}],
 'meta':

In [24]:
# Convert result to Pandas DataFrame
results_df = pd.DataFrame(hygiene_results)
results_df.head()

""


### 4. How many establishments in each Local Authority area have a hygiene score of 0?

In [29]:
# Create a pipeline that:
# 1. Matches establishments with a hygiene score of 0
# 2. Groups the matches by Local Authority
# 3. Sorts the matches from highest to lowest
parameters = [
    {'$match': {'scores.Hygiene': 0}},
    {'$group': {
        '_id': '$LocalAuthorityName',
        'count': {'$sum': 1} 
    }},
    {'$sort': {'count': -1}}
]

score_zero_results = list(establishments.aggregate(parameters))
# Print the number of documents in the result
print('Number of Documents:', len(score_zero_results))
# Print the first 10 results
for x in range(min(10, len(score_zero_results))):
    print(score_zero_results[x])

Number of Documents: 55
{'_id': 'Thanet', 'count': 1130}
{'_id': 'Greenwich', 'count': 882}
{'_id': 'Maidstone', 'count': 713}
{'_id': 'Newham', 'count': 711}
{'_id': 'Swale', 'count': 686}
{'_id': 'Chelmsford', 'count': 680}
{'_id': 'Medway', 'count': 672}
{'_id': 'Bexley', 'count': 607}
{'_id': 'Southend-On-Sea', 'count': 586}
{'_id': 'Tendring', 'count': 542}


In [30]:
# Convert the result to a Pandas DataFrame
zeroresults_df = pd.DataFrame(score_zero_results)
# Display the number of rows in the DataFrame
print('Number of Rows:', len(results_df))
# Display the first 10 rows of the DataFrame
zeroresults_df.head(10)

Number of Rows: 0


,_id,count
0,Thanet,1130
1,Greenwich,882
2,Maidstone,713
3,Newham,711
4,Swale,686
5,Chelmsford,680
6,Medway,672
7,Bexley,607
8,Southend-On-Sea,586
9,Tendring,542
